# Summary

1. Load data from ```process_uber_summary``` table and build **all process (sub)trees of size 3 or more**
2. Load PID's embedding obtained from numerical features (see notebook ```process_embeddings```)
3. Run our **fast matching** algorithm using:
   * process_name only
   * append cluster number from numerical features
   * actually use distance between numerical feature embeddings

### References

* https://github.com/llnl/Wintap-Analytics/tree/main/2025-acme4-explore
* https://gdo168.llnl.gov/
* https://gdo168.llnl.gov/data/newdocs/datadict/
  

In [ ]:
%%time
import io
import logging as lg
import numpy as np
import os
import pandas as pd
import re
from collections import Counter
import warnings
import pickle
import gzip

# Modelling
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import adjusted_mutual_info_score as AMI
from scipy.spatial import distance
from sklearn.neighbors import NearestNeighbors
import random 

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz
import igraph as ig
import partition_igraph
import umap

# utility functions and matching algorithm
## our matching algorithm
import sys
import os
sys.path.insert(0,os.path.abspath('../Matching'))
import process_trees as pt
import igraph_io as igio
import gw_matcher as gwm ## older code - for testing
import fast_match as fm  ## fast version


# Generate or load process tree objects


In [ ]:
## load df_trees and Trees
with gzip.open('Data/acme4_process_trees_and_df.pkl.gz', 'rb') as fp:
   Trees, df_trees =  pickle.load(fp)


#### Generate the **Trees** and related **df_trees** objects

Uncomment the cells below to generate; takes over 30 min


In [ ]:
## only 'baduser' are colored red by default - add 'redteam' positive labels
for v in Trees.graph.vs:
    if v['redteam']==1:
        v['label_color']='red'

### load embeddings (based on numerical features) 

Those are dictionaries with key=PID, value=embedding as an array, built in notebook ```process_embedding.ipynb```


In [ ]:
with gzip.open('Data/acme4_process_num_features_embedding_ecdf.pkl.gz', 'rb') as fp:
    Embedding_ecdf = pickle.load(fp)


### Add inverse frequency score

* We will also compute an inverse-frequency score as post-processing for the matching algorithm (sum of inverse frequency of matched processes)
* Useful to find matchings over "less common" processes


In [ ]:
proc_counts = Counter(Trees.graph.vs['process'])
InvFreq = {k: 1/np.log2(v+1) for k, v in proc_counts.items()} ## max score of 1 when v==1


# Matching algorithm

## (1) Find best subtree match - baduser trees vs non-baduser trees


In [ ]:
Trees.graph.vs['bad'] = [x is not None and 'bad' in x for x in Trees.graph.vs['username']]

_ = df_trees[(df_trees.nodes<=1500) & (df_trees.layers>=8) & (df_trees.badusers>0)]
bad_trees = [(list(_.tree)[i],list(_.root)[i]) for i in range(_.shape[0])]
_ = df_trees[(df_trees.nodes<=1500) & (df_trees.layers>=8) & (df_trees.badusers==0)]
nonbad_trees = [(i,j) for i,j in zip(list(_.tree), list(_.root))]


In [ ]:
%%time

feature_to_match = 'process'

## build similarity matrix between all tree pairs
Graphs = []
TreeData = []
len_bad = len(bad_trees)
len_nonbad = len(nonbad_trees)
Sim = np.zeros(shape=(len_bad, len_nonbad))
Score =  np.zeros(shape=(len_bad, len_nonbad))
dim = len(next(iter(Embedding_ecdf.values())))

## list all bad graphs
for tree,root in bad_trees:
    sg = pt.get_bfs_subtree(Trees, tree, root) 
    Graphs.append(sg)
    TreeData.append(igio.igraph_to_treedata(sg, phi_name=feature_to_match ))
## add nonbad trees
for tree,root in nonbad_trees:
    sg = pt.get_bfs_subtree(Trees, tree, root) 
    Graphs.append(sg)
    TreeData.append(igio.igraph_to_treedata(sg, phi_name=feature_to_match ))

## encode all trees
fast = fm.FastTreePathMatcher()
fast.fit_encoder(TreeData)
enc = [fast.encode_tree(t) for t in TreeData]

for i in range(len_bad):
    for j in range(len_nonbad):
        paths, score = fast.predict_encoded(enc[i], enc[j+len_bad])
        p = [x[0] for x in paths]
        score = sum([InvFreq[ Graphs[i].vs[k][feature_to_match] ] for k in p])
        if sum([Graphs[i].vs[k]['bad'] for k in p])==0:
            score = 0
        Score[i,j] = score
        if score>0:
            sim = np.sum([1-distance.cityblock( Embedding_ecdf[Graphs[i].vs[x[0]]['pid']] , Embedding_ecdf[Graphs[j+len_bad].vs[x[1]]['pid']] )/dim for x in paths])
        else:
            sim = 0
        Sim[i,j] = sim
        

### top match w.r.t. inverse frequency score

In [ ]:
## pick top similarity pair and visualize common path
(bad, nonbad) = np.unravel_index(np.argmax(Score), Sim.shape)
sg1 = pt.get_bfs_subtree(Trees, bad_trees[bad][0], bad_trees[bad][1])
sg2 = pt.get_bfs_subtree(Trees, nonbad_trees[nonbad][0], nonbad_trees[nonbad][1])
_sg2 = igio.igraph_to_treedata(sg2, phi_name='process')
_sg1 = igio.igraph_to_treedata(sg1, phi_name='process')
fast = fm.FastTreePathMatcher()
fast.fit(_sg1,_sg2)
paths, score = fast.predict()
print('length of matching path:',score)
print('embedded space similarity score:',np.max(Sim))
print('inverse frequency score:',Score[bad,nonbad])


In [ ]:
## plot tree with baduser ; highlight path
path = [int(x[0]) for x in paths]
sg1.vs['vertex_size'] = 1
sg1.vs['vertex_label_size'] = 1
for i in path:
    sg1.vs[i]['vertex_size'] = 1
    sg1.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg1.vcount())
ig.plot(sg1, 
        bbox=(1000,600), layout=sg1['ly'], margin=100, 
        vertex_size=sg1.vs['vertex_size'], 
        vertex_label=sg1.vs['label'], 
        vertex_label_size=sg1.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


In [ ]:
## plot top scoring tree
path = [int(x[1]) for x in paths]
sg2.vs['vertex_size'] = 1
sg2.vs['vertex_label_size'] = 0
for i in path:
    sg2.vs[i]['vertex_size'] = 1
    sg2.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg2.vcount())
ig.plot(sg2,
        bbox=(1000,800), layout=sg2['ly'], margin=200, 
        vertex_size=sg2.vs['vertex_size'], 
        vertex_label=sg2.vs['label'], 
        vertex_label_size=sg2.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


In [ ]:
## low frequency processes indeed
p = [sg2.vs[i]['process'] for i in path]
_dct = Counter(Trees.graph.vs['process'])
[_dct[i] for i in p]


### top match w.r.t. embedded space similarity score

In [ ]:
## pick top similarity pair and visualize common path
(bad, nonbad) = np.unravel_index(np.argmax(Sim), Sim.shape)
sg1 = pt.get_bfs_subtree(Trees, bad_trees[bad][0], bad_trees[bad][1])
sg2 = pt.get_bfs_subtree(Trees, nonbad_trees[nonbad][0], nonbad_trees[nonbad][1])
_sg2 = igio.igraph_to_treedata(sg2, phi_name='process')
_sg1 = igio.igraph_to_treedata(sg1, phi_name='process')
fast = fm.FastTreePathMatcher()
fast.fit(_sg1,_sg2)
paths, score = fast.predict()
print('length of matching path:',score)
print('embedded space similarity score:',np.max(Sim))
print('inverse frequency score:',Score[bad,nonbad])


In [ ]:
## plot tree with baduser ; highlight path
path = [int(x[0]) for x in paths]
sg1.vs['vertex_size'] = 1
sg1.vs['vertex_label_size'] = 1
for i in path:
    sg1.vs[i]['vertex_size'] = 1
    sg1.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg1.vcount())
ig.plot(sg1, 
        bbox=(1000,600), layout=sg1['ly'], margin=100, 
        vertex_size=sg1.vs['vertex_size'], 
        vertex_label=sg1.vs['label'], 
        vertex_label_size=sg1.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


In [ ]:
## plot top scoring tree
path = [int(x[1]) for x in paths]
sg2.vs['vertex_size'] = 1
sg2.vs['vertex_label_size'] = 0
for i in path:
    sg2.vs[i]['vertex_size'] = 1
    sg2.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg2.vcount())
ig.plot(sg2,'match_nonbad_embed.png', 
        bbox=(1000,800), layout=sg2['ly'], margin=100, 
        vertex_size=sg2.vs['vertex_size'], 
        vertex_label=sg2.vs['label'], 
        vertex_label_size=sg2.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


In [ ]:
path0 = [int(x[0]) for x in paths]
path1 = [int(x[1]) for x in paths]
l = len(path0)
c = np.repeat([0,1],l)
s = np.tile(np.arange(l),2)
X = []
for i in path0:
    X.append(Embedding_ecdf[sg1.vs[i]['pid']])
for i in path1:
    X.append(Embedding_ecdf[sg2.vs[i]['pid']])

## 2d projection
reducer = umap.UMAP(n_neighbors=3, metric='manhattan', min_dist=.25, spread=1.5)
embedding = reducer.fit_transform(np.array(X))


In [ ]:
# plot it - color w.r.t. response variable (bad/nonbad)
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(embedding[:, 0], embedding[:, 1], s=10, c=c, cmap='cool')
ax.set_title('ecdf embedding')
for i in range(embedding.shape[0]):
    ax.annotate(s[i], (embedding[i, 0], embedding[i, 1]))
plt.savefig('embedded_space.png')
plt.show()


## (2) Enrich labels via clustering

* clusters in embedded space


In [ ]:
## get embeddings 
A = np.array(list(Embedding_ecdf.values()))
avg_emb = np.mean(A, axis=0)
Pids = Trees.graph.vs['pid']
E = np.array([Embedding_ecdf.get(x,avg_emb) for x in Pids]) ## some roots (parents only) have no embedding


In [ ]:
## fast clustering
from sklearn.cluster import MiniBatchKMeans
mbk = MiniBatchKMeans(n_clusters=20, batch_size=1024, n_init="auto", random_state=42)
mbk.fit(E)
cluster_dct = dict(zip(Pids,mbk.labels_))
G = Trees.graph
G.vs['cluster'] = [str(i) for i in mbk.labels_] 


In [ ]:
## measure of association
X = G.vs['cluster']
#Y = G.vs['bh']
Z = G.vs['process']
#print(f"Cramer's V (cluster,bh): {pt.cramers_v(X, Y)}")
print(f"Cramer's V (cluster,process): {pt.cramers_v(X, Z)}")
#print(f"Cramer's V (bh,process): {pt.cramers_v(Y, Z)}")


In [ ]:
## add enriched label process+cluster
G.vs['enriched'] = [f"{a}_{b}" for a, b in zip(G.vs['process'], G.vs['cluster'])]
G.vs['enriched_label'] = [f"{a}::{b}" for a, b in zip(G.vs['shortlabel'], G.vs['enriched'])]


In [ ]:
%%time

feature_to_match = 'enriched'

## build similarity matrix between all tree pairs
Graphs = []
TreeData = []
len_bad = len(bad_trees)
len_nonbad = len(nonbad_trees)
Sim = np.zeros(shape=(len_bad, len_nonbad))
Score =  np.zeros(shape=(len_bad, len_nonbad))
dim = len(next(iter(Embedding_ecdf.values())))

## list all bad graphs
for tree,root in bad_trees:
    sg = pt.get_bfs_subtree(Trees, tree, root) 
    Graphs.append(sg)
    TreeData.append(igio.igraph_to_treedata(sg, phi_name=feature_to_match ))
## add nonbad trees
for tree,root in nonbad_trees:
    sg = pt.get_bfs_subtree(Trees, tree, root) 
    Graphs.append(sg)
    TreeData.append(igio.igraph_to_treedata(sg, phi_name=feature_to_match ))

## encode all trees
fast = fm.FastTreePathMatcher()
fast.fit_encoder(TreeData)
enc = [fast.encode_tree(t) for t in TreeData]

for i in range(len_bad):
    for j in range(len_nonbad):
        paths, score = fast.predict_encoded(enc[i], enc[j+len_bad])
        p = [x[0] for x in paths]
        score = sum([InvFreq[ Graphs[i].vs[k]['process'] ] for k in p])
        if sum([Graphs[i].vs[k]['bad'] for k in p])==0:
            score = 0
        Score[i,j] = score
        if score>0:
            sim = np.sum([1-distance.cityblock( Embedding_ecdf[Graphs[i].vs[x[0]]['pid']] , Embedding_ecdf[Graphs[j+len_bad].vs[x[1]]['pid']] )/dim for x in paths])
        else:
            sim = 0
        Sim[i,j] = sim
        

In [ ]:
## pick top similarity pair and visualize common path
(bad, nonbad) = np.unravel_index(np.argmax(Sim), Sim.shape)
sg1 = pt.get_bfs_subtree(Trees, bad_trees[bad][0], bad_trees[bad][1])
sg2 = pt.get_bfs_subtree(Trees, nonbad_trees[nonbad][0], nonbad_trees[nonbad][1])
_sg2 = igio.igraph_to_treedata(sg2, phi_name='enriched')
_sg1 = igio.igraph_to_treedata(sg1, phi_name='enriched')
fast = fm.FastTreePathMatcher()
fast.fit(_sg1,_sg2)
paths, score = fast.predict()
print('length of matching path:',score)
print('embedded space similarity score:',np.max(Sim))
print('inverse frequency score:',Score[bad,nonbad])


In [ ]:
## plot tree with baduser ; highlight path
path = [int(x[0]) for x in paths]
sg1.vs['vertex_size'] = 1
sg1.vs['vertex_label_size'] = 1
for i in path:
    sg1.vs[i]['vertex_size'] = 1
    sg1.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg1.vcount())
ig.plot(sg1,
        bbox=(1000,600), layout=sg1['ly'], margin=100, 
        vertex_size=sg1.vs['vertex_size'], 
        vertex_label=sg1.vs['enriched_label'], 
        vertex_label_size=sg1.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


In [ ]:
## plot top scoring tree
path = [int(x[1]) for x in paths]
sg2.vs['vertex_size'] = 1
sg2.vs['vertex_label_size'] = 0
for i in path:
    sg2.vs[i]['vertex_size'] = 1
    sg2.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg2.vcount())
ig.plot(sg2,
        bbox=(1000,800), layout=sg2['ly'], margin=100, 
        vertex_size=sg2.vs['vertex_size'], 
        vertex_label=sg2.vs['enriched_label'], 
        vertex_label_size=sg2.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


## (3) Using several templates

We run a classifiction experiment on common root process names

In [ ]:
### filter subtrees 
_df = df_trees[(df_trees.nodes<=5000) & (df_trees.layers>=3)]
Counter(_df.process).most_common(10)


In [ ]:
n_class = 6
n_templates = 100

_df = df_trees[(df_trees.nodes<=5000) & (df_trees.layers>=3)]
common_proc = [x[0] for x in Counter(_df.process).most_common(n_class)]
_df = _df[_df.process.isin(common_proc)]

## random templates
min_rep = 0
np.random.seed(42)
while min_rep < 10:
    r = np.random.choice(_df.shape[0], n_templates, replace=False)
    proc = [list(_df.process)[i] for i in r]
    if len(set(proc)) < n_class:
        continue        
    min_rep = min(Counter(proc).values())
templates = [(list(_df.tree)[i],list(_df.root)[i]) for i in r]
Counter(proc)


In [ ]:
%%time
## build similarity matrix between all trees and templates
Graphs = []
TreeData = []
T = list(_df.tree)
R = list(_df.root)
l = len(templates)
Sim = np.zeros(shape=(l, len(T)))
Score = np.zeros(shape=(l, len(T)))
dim = len(next(iter(Embedding_ecdf.values())))

## list all template graphs
for tree,root in templates:
    sg1 = pt.get_bfs_subtree(Trees, tree, root) ## sub-tree above with required DFS labelling
    Graphs.append(sg1)
    TreeData.append(igio.igraph_to_treedata(sg1, phi_name='process'))
## add trees to be compared 
for j in range(len(T)):
    sg2 = pt.get_bfs_subtree(Trees, T[j], R[j])
    Graphs.append(sg2)
    TreeData.append(igio.igraph_to_treedata(sg2, phi_name='process'))
print('number of matches:', l*len(T))


In [ ]:
%%time
## encode all trees
fast = fm.FastTreePathMatcher()
fast.fit_encoder(TreeData)
enc = [fast.encode_tree(t) for t in TreeData]

for i in range(l):
    for j in range(len(T)):
        paths, score = fast.predict_encoded(enc[i], enc[j+l])
        Score[i,j] = score
        if score>0:
            # print(i, j, paths)
            # sim = np.mean([1-distance.cityblock( Embedding_ecdf[Graphs[i].vs[x[0]]['pid']] , Embedding_ecdf[Graphs[j+l].vs[x[1]]['pid']] )/dim for x in paths])
            sim = np.sum([1-distance.cityblock( Embedding_ecdf[Graphs[i].vs[x[0]]['pid']] , Embedding_ecdf[Graphs[j+l].vs[x[1]]['pid']] )/dim for x in paths])
        else:
            sim = 0
        Sim[i,j] = sim

In [ ]:
label_map = {v:k for k,v in enumerate(common_proc)}
X = Score.T.tolist()
y = [label_map[i] for i in _df.process]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=123)
rf = RandomForestClassifier(n_jobs=16, random_state=123)
rf.fit(X_train, y_train)


In [ ]:
## confusion matrix
y_pred = rf.predict(X_test)
cm = confusion_matrix(y_test,y_pred)
fig,ax = plt.subplots(figsize=(9,7))
plt.rcParams.update({'font.size': 14}) # Change to your desired size
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                              display_labels=list(label_map.keys())
                             )
disp.plot(colorbar=False, cmap='Blues', ax=ax)
plt.xticks(fontsize=10)
plt.yticks(rotation=45, fontsize=10)
plt.show()


In [ ]:
label_map = {v:k for k,v in enumerate(common_proc)}
#X = ((Score+1)*Sim).T.tolist()
X = (Sim).T.tolist()
y = [label_map[i] for i in _df.process]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=123)
rf = RandomForestClassifier(n_jobs=16, random_state=123)
rf.fit(X_train, y_train)


In [ ]:
## confusion matrix
y_pred = rf.predict(X_test)
cm = confusion_matrix(y_test,y_pred)
fig,ax = plt.subplots(figsize=(9,7))
plt.rcParams.update({'font.size': 14}) # Change to your desired size
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                              display_labels=list(label_map.keys())
                             )
disp.plot(colorbar=False, cmap='Blues', ax=ax)
plt.xticks(fontsize=10)
plt.yticks(rotation=45, fontsize=10)
#plt.savefig('confusion.png')
plt.show()


In [ ]:
## root processes' embeddings
E = []
C = []
for t,r in zip(list(_df.tree),list(_df.root)):
    v = Trees.subgraph(t).vs[r]
    E.append(Embedding_ecdf[v['pid']])
    C.append(v['process'])
Clusters = np.array(C)
Embs = np.array(E)


In [ ]:
## plot
reducer = umap.UMAP(n_neighbors=15, metric='manhattan', init='random', random_state=123, n_jobs=1)
embedding = reducer.fit_transform(E)



In [ ]:
# plot it - color w.r.t. response variable (bad/nonbad)
plt.style.use('dark_background')
plt.figure(figsize=(10,8))
_dct = {v:k for k,v in enumerate(set(C))}
colors = ['#FFC20A','#28A1C7','#801650','#F46A25','#DC3220','#3cb44b']

for c, l in enumerate(_dct):
    plt.scatter(embedding[Clusters==l, 0], embedding[Clusters==l, 1], s=10, c=colors[c], label=l, alpha=.65)
plt.title('ecdf embedding')
leg = plt.legend()
for handle in leg.legend_handles:
    handle.set_alpha(1.0)
plt.show()


### kNN and graph clustering with ECG

In [ ]:
## scaled mixed score 
scaler = StandardScaler()
X = scaler.fit_transform( (Score+Sim).T )

# fit
knn = NearestNeighbors(n_neighbors=11, metric='minkowski', p=2)
knn.fit(X)
K = knn.kneighbors(X)[1]


In [ ]:
## plot
reducer = umap.UMAP(n_neighbors=11, metric='manhattan', init='random', random_state=123, n_jobs=1)
embedding = reducer.fit_transform(X)


In [ ]:
# plot it - color w.r.t. response variable (bad/nonbad)
plt.style.use('dark_background')
plt.figure(figsize=(10,8))
_dct = {v:k for k,v in enumerate(set(C))}
colors = ['#FFC20A','#28A1C7','#801650','#F46A25','#DC3220','#3cb44b']

for c, l in enumerate(_dct):
    plt.scatter(embedding[Clusters==l, 0], embedding[Clusters==l, 1], s=10, c=colors[c], label=l, alpha=.65)
plt.title('UMAP projection')
leg = plt.legend()
for handle in leg.legend_handles:
    handle.set_alpha(1.0)
#plt.show()
plt.savefig('umap_proj.png')


In [ ]:
## build graph
_dict = {i: row for i, row in enumerate(K)}
G = ig.Graph.ListDict(_dict, )
G = G.simplify()
G.vs['label'] = [str(i) for i in Clusters] ## Clusters from previous experiment
G.summary()


In [ ]:
## ECG clustering
np.random.seed(42)
random.seed(42)
GCC = G.community_ecg(ens_size=100, resolution=.01, final='leiden')
G.es['W'] = GCC.W
print(GCC.summary())
print('AMI:',AMI(GCC.membership, G.vs['label']))


In [ ]:
## add colors based on root process
_dct = {v:k for k,v in enumerate(set(Clusters))}
colors = ['#FFC20A','#28A1C7','#801650','#F46A25','#DC3220','#3cb44b']
G.vs['color'] = [colors[_dct[i]] for i in Clusters]

## plot with GT colors
random.seed(42)
ly = G.layout_fruchterman_reingold()
#ly = G.layout_kamada_kawai()
ig.plot(G, target='knn_graph.png', vertex_size=8, vertex_label_size=0, edge_color='grey', layout=ly)


In [ ]:
## plot
cmap = plt.get_cmap('tab20')
colors = [plt.cm.colors.to_hex(cmap(i)) for i in range(max(GCC.membership)+1)]
G.es['color'] = 'grey'
#G.es['width'] = [1 if e['w']>0 else 0 for e in G.es ]
ig.plot(GCC, vertex_size=5, vertex_label_size=0, layout=ly, vertex_color=[colors[i] for i in GCC.membership], mark_groups=True)
